<a href="https://colab.research.google.com/github/qndks11/cv-playground/blob/main/CIFAR10_ResNET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pytorch 다운로드

In [ ]:
import torch
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BATCH_SIZE = 32
EPOCHS = 25

# CIFAR 10 다운로드

In [ ]:
# Define a transform to normalize the data
transform = transforms.Compose([
    transforms.ToTensor(),  # [0, 255] → [0, 1], HWC → CHW
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]       # 채널별로 평균을 0, 분산을 1로 정규화
    )
])

# 트레이닝 데이터 다운로드
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
# 테스트 데이터 다운로드
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size = BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size = BATCH_SIZE, shuffle=True)

# ResNet

ResNet의 핵심 아이디어는 "identity mapping을 직접 학습하는 대신, residual F(x)을 학습해서 0으로 수렴시키자"는 것이다.

상식적으로 생각했을 때 Layer를 더 쌓으면 성능이 개선되거나 최소한 유지되어야 한다. 예를 들어 4개의 Layer가 있을 때, 레이어를 하나 더 추가하고 그 레이어가 Input을 그대로 Output으로 내보내게 한다면, 성능은 최소한 그대로여야 한다.

하지만 실제로는 Layer가 추가될 때 오히려 성능이 떨어지는 현상이 발생한다. Overfitting 때문도 아니다. Training Loss 자체가 올라가기 때문이다.

원인은 ReLU와 conv 레이어를 쌓아서 identity 함수를 정확히 만들어내는 것이 최적화 관점에서 어렵기 때문이다. 반면 출력을 0으로 만드는 것은 쉽다. 모든 weight를 0으로 두면 되기 때문이다.

그래서 ResNet은 H(x)를 직접 학습하는 대신, H(x) = F(x) + x 형태로 구조를 바꾸고 F(x)를 0으로 수렴시키도록 학습시킨다. F(x)가 0이 되면 자동으로 H(x) = x가 되므로, identity mapping이라는 어려운 목표를 "weight를 0으로 만드는" 쉬운 목표로 바꿔준 것이다.

In [ ]:
class BasicBlock(torch.nn.Module):
  def __init__(self, in, out):
    super(CNN, self).__init__()
    self.conv1 = torch.nn.Conv2d(in_channels=in, out_channels=out, kernel_size=3, stride=1)

  def forward(self, x):
    x = self.conv1(x)
    x = self.relu(x)
    x = self.pool(x)
    x = self.conv2(x)
    x = self.relu(x)
    x = self.pool(x)

    x = x.view(x.size(0), -1)
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    return x

class ResNET(torch.nn.Module):
  def __init__(self):
    super(CNN, self).__init__()
    self.conv1 = torch.nn.Conv2d(in_channels=1, out_channels=16, kernel_size=5, stride=1)
    self.conv2 = torch.nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1)
    self.pool = torch.nn.MaxPool2d(kernel_size=2)
    self.relu = torch.nn.ReLU(inplace=True)

    self.fc1 = torch.nn.Linear(32*5*5, 64)
    self.fc2 = torch.nn.Linear(64, 10)

  def forward(self, x):
    x = self.conv1(x)
    x = self.relu(x)
    x = self.pool(x)
    x = self.conv2(x)
    x = self.relu(x)
    x = self.pool(x)

    x = x.view(x.size(0), -1)
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    return x